# Validation results

_Authors: Andreia Dourado, Bruno Moraes_

_Adapted from the notebooks https://github.com/LSSTDESC/rail_tpz e https://rail-hub.readthedocs.io/projects/rail-notebooks/en/latest/rendered/evaluation_examples/Evaluation_Demo.html_

__Description: Analysis of the metrics for the results generated in the Estimate step for TPZ.__

### 1. Imports:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rail
import qp
import tables_io
from rail.core.data import TableHandle, PqHandle, ModelHandle, QPHandle, DataHandle, Hdf5Handle
from rail.core.stage import RailStage

In [ ]:
from qp import Ensemble
from matplotlib import gridspec
from qp import interp
from qp.metrics.pit import PIT
from rail.evaluation.metrics.cdeloss import *
from rail.evaluation.evaluator import OldEvaluator
from rail.evaluation.point_to_point_evaluator import PointToPointEvaluator
from rail.estimation.algos.point_est_hist import PointEstHistSummarizer
from rail.evaluation.metrics.cdeloss import *
from utils import plot_pit_qq, ks_plot
import os
from rail.estimation.algos.naive_stack import NaiveStackSummarizer
from scipy.interpolate import UnivariateSpline


%matplotlib inline
%reload_ext autoreload
%autoreload 

### 2. Reading the data

In [ ]:
path_true = '../run_files/'

In [ ]:
path_mode = '../output/'

#### 2.1 Test and output files

In [ ]:
ztrue_file= f'{path_true}test_file.hdf5'
ztrue_data = tables_io.read(ztrue_file)

In [ ]:
pdfs_file=f'{path_mode}/output_tpz_10nrandom_10ntree_20minleaf_3att_test.hdf5'
tpzdata = tables_io.read(pdfs_file)

#### 2.2 Multiple files:

In [ ]:
pdfs_file1=f'{path_mode}/output_test_mags+colors_minleaf30.hdf5'
tpzdata1 = tables_io.read('pdfs_data', QPHpdfs_file1)

zgrid1 = tpzdata1.data[0].gen_obj.xvals
photoz_mode1 = tpzdata1().mode(grid=zgrid1)
z_mode1= np.squeeze(photoz_mode1)

In [ ]:
pdfs_file2=f'{path_mode}/output_test_mags_minleaf30.hdf5'
tpzdata2 = DS.read_file('pdfs_data', QPHandle, pdfs_file2)

zgrid2 = tpzdata2.data[0].gen_obj.xvals
photoz_mode2 = tpzdata2().mode(grid=zgrid2)
z_mode2= np.squeeze(photoz_mode2)

In [ ]:
pdfs_file3=f'{path_mode}/output/output_test_colors_minleaf30.hdf5'
tpzdata3 = DS.read_file('pdfs_data', QPHandle, pdfs_file3)

zgrid3 = tpzdata3.data[0].gen_obj.xvals
photoz_mode3 = tpzdata3().mode(grid=zgrid3)
z_mode3= np.squeeze(photoz_mode3)

In [ ]:
ztrue = ztrue_data()['redshift']

In [ ]:
zgrid = np.linspace(0, 3., 301)

In [ ]:
len(ztrue), len(z_mode1), len(z_mode2), len(z_mode3)

#### 2.3 Reading the true and estimated redshift values:

In [ ]:
ztrue = ztrue_data['photometry']['redshift']
zgrid = np.linspace(0,3,301)
#photoz_mode = tpzdata().mode(grid=zgrid)
z_mode= tpzdata['ancil']['zmode'].ravel()

In [ ]:
#truth = DS.add_data('truth', ztrue_data(), TableHandle)
#ensemble = DS.add_data('ensemble', tpzdata(), QPHandle)

In [ ]:
ensemble = QPHandle("pdfs_data",path=pdfs_file)
truth = TableHandle("ztrue_data", path=ztrue_file)

In [ ]:
len(z_mode), len(ztrue)

### 3. Metrics

__Path to save images:__

In [ ]:
path = '../metricas/'

#### 3.1 Zphot x Ztrue

In [ ]:
def plot_scatter(zphot, ztrue, zmin=0, zmax=3):

    h = sns.histplot(x=ztrue, y=zphot, bins=150, cmap='viridis')
    plt.plot([0,3], [0,3], color='red', linewidth=0.2)
    plt.xlim(zmin, zmax)
    plt.ylim(zmin, zmax)
    plt.xlabel('z$_{true}$', fontsize=20)
    plt.ylabel('z$_{phot}$', fontsize=20)
    plt.colorbar(h.collections[0], label='counts')
    plt.tick_params(axis='both', labelsize=12)


    
    plt.savefig(f'{path}scatter_mags.pdf', format='pdf', bbox_inches='tight', dpi=300)
    
    plt.show()

In [ ]:
plot_scatter(z_mode,ztrue)

#### 3.2. Individual PDF

In [ ]:
numeros = [16391,16180,27054,4773,27937,23796,27034,347,22830]

In [ ]:
print(numeros)
j=1
fig, axs = plt.subplots(3, 3, figsize=(20, 15))
axs = axs.flatten() 

for j, i in enumerate(numeros):
    which= i
    ax = axs[j]
    ensemble().plot_native(key=which,axes=ax, label=f"PDF Galáxia {j+1}")
    ax.axvline(ztrue[which],c='r',ls='--', label=f"spec-z = {(ztrue[which]).round(2)}")
    ax.axvline(z_mode[which],c='black',ls='--', label=f"photo-z mode = {z_mode[which]}")
    ax.legend(loc='upper right', fontsize=15)
    ax.set_xlabel("redshift", fontsize=20)
    ax.tick_params(axis='both', labelsize=17)
    #ax.set_title(f"Galáxia {j+1}", fontsize=20)
    j+=1
    

plt.tight_layout()
plt.savefig(f'{path}pdfs_example.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

#### 3.3 Metrics

##### Functions

In [ ]:
def compute_photoz_metrics(zspec, zphot):
    
    delta_z = (zphot - zspec) / (1 + zspec)
    

    rms = np.sqrt(np.mean(delta_z**2))
    
    bias = np.mean(delta_z)

    sigma = np.std(delta_z)
    #sigma = np.sqrt(np.mean((delta_z - bias)**2))

    lower = np.percentile(delta_z, 15.87)
    upper = np.percentile(delta_z, 84.13)
    sigma_68 = 0.5 * (upper - lower)

    out_2sigma = np.sum(np.abs(delta_z) > 2 * sigma) / len(delta_z)

    out_3sigma = np.sum(np.abs(delta_z) > 3 * sigma) / len(delta_z)


    return {
        'bias': bias,
        'sigma_68': sigma_68,
        'sigma': sigma,
        'out_2sigma': out_2sigma,
        'out_3sigma': out_3sigma,
        'RMS': rms
    }

In [ ]:
def plot_metrics_bias_sep(zspec, zphot, maximum, path_to_save='', title=None, initial=0):
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    sns.set_context("paper", font_scale=1.3)
    sns.set_style("whitegrid")
    plt.rcParams.update({
        "font.family": "serif",
        "axes.edgecolor": "black",
        "axes.linewidth": 1.2,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": 5,
        "ytick.major.size": 5
    })

    bins = np.arange(initial, maximum, 0.1)
    points = bins + 0.05

    bias_list = []
    sigma_list = []
    sigma68_list = []
    out2_list = []
    out3_list = []

    for i in range(len(bins) - 1):
        zmin, zmax = bins[i], bins[i + 1]
        mask = (zphot >= zmin) & (zphot < zmax)
        zp, zs = zphot[mask], zspec[mask]

        if len(zp) == 0:
            bias_list.append(np.nan)
            sigma_list.append(np.nan)
            sigma68_list.append(np.nan)
            out2_list.append(np.nan)
            out3_list.append(np.nan)
            continue

        dz = zp - zs
        bias = np.mean(dz / (1 + zs))
        sigma = np.std(dz / (1 + zs))
        sorted_dz = np.sort(np.abs(dz / (1 + zs)))
        sigma68 = sorted_dz[int(len(sorted_dz) * 0.68)]
        out2 = np.mean(np.abs(dz - bias) > 2 * sigma)
        out3 = np.mean(np.abs(dz - bias) > 3 * sigma)

        bias_list.append(bias)
        sigma_list.append(sigma)
        sigma68_list.append(sigma68)
        out2_list.append(out2)
        out3_list.append(out3)

    fig, (ax_out, ax_sigma, ax_bias) = plt.subplots(3, 1, figsize=(10, 12), sharex=True, gridspec_kw={'height_ratios': [1, 1, 1]})
    plt.subplots_adjust(hspace=0.1)

    ax_sigma.plot(points[:-1], sigma68_list, 'o-', label=r'$\sigma_{68}$', color='forestgreen')
    ax_sigma.set_ylabel(r'$\sigma_{68}$', fontsize=18)
    ax_sigma.set_xlim(initial, 2)
    ax_sigma.legend(fontsize=14)
    ax_sigma.tick_params(axis='both', labelsize=12)
    ax_sigma.grid(True)

    ax_out.plot(points[:-1], out2_list, 'o-', label=r'Outliers 2$\sigma$', color='darkorange')
    ax_out.plot(points[:-1], out3_list, 'o-', label=r'Outliers 3$\sigma$', color='crimson')
    ax_out.set_ylabel("Outliers", fontsize=18)
    ax_out.set_xlim(initial, 2)
    ax_out.legend(loc='upper left', fontsize=14)
    ax_out.tick_params(axis='both', labelsize=12)
    ax_out.grid(True)

    ax_bias.plot(points[:-1], bias_list, 'o-', color='royalblue', label='Bias (Δz)')
    ax_bias.fill_between(points[:-1],
                         np.array(bias_list) - np.array(sigma_list),
                         np.array(bias_list) + np.array(sigma_list),
                         color='royalblue', alpha=0.2, label='±1σ')
    ax_bias.axhline(0, linestyle='--', color='gray', lw=1)
    ax_bias.set_xlabel(r'$z_{\mathrm{spec}}$', fontsize=18)
    ax_bias.set_ylabel(r'$\Delta z$', fontsize=18)
    ax_bias.set_xlim(initial, 2)
    ax_bias.grid(True)
    ax_bias.legend(loc='upper left', fontsize=14)
    ax_bias.tick_params(axis='both', labelsize=12)

    if title:
        fig.suptitle(title, fontsize=20)

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    plt.savefig(f'{path}metrics.pdf', format='pdf', dpi=300, bbox_inches='tight')
    plt.show()


##### Plots

In [ ]:
plot_metrics_bias_sep(ztrue, z_mode, max(z_mode))

In [ ]:
compute_photoz_metrics(ztrue,z_mode)

#### 3.4 PIT QQ

In [ ]:
pitobj = PIT(ensemble(), ztrue)
quant_ens = pitobj.pit
metamets = pitobj.calculate_pit_meta_metrics()

In [ ]:
metamets

In [ ]:
pit_vals = np.array(pitobj.pit_samps)
pit_vals

In [ ]:
pit_out_rate = metamets['outlier_rate']
print(f"PIT outlier rate of this sample: {pit_out_rate:.6f}")
pit_out_rate = pitobj.evaluate_PIT_outlier_rate()
print(f"PIT outlier rate of this sample: {pit_out_rate:.6f}")

In [ ]:
pdfs = ensemble.data.objdata['yvals']

In [ ]:
from utils import plot_pit_qq

plot_pit_qq(pdfs, zgrid, ztrue, title="PIT-QQ", code="TPZ",
                pit_out_rate=pit_out_rate, savefig=False)
plt.grid(False)
plt.savefig(f'{path}pitqq.pdf', format='pdf', dpi=300, bbox_inches='tight')

#### 3.5 N(z)

In [ ]:
stacker = NaiveStackSummarizer.make_stage(zmin=0.0, zmax=3, nzbins=301, nsamples=20, hdf5_groupname=None, output=f"Naive_sample.hdf5", single_NZ=f"NaiveStack_TPZ.hdf5")

In [ ]:
naive_results = stacker.summarize(ensemble)

In [ ]:
fig = plt.figure(figsize=(10, 8))


plt.xlabel('redshift', fontsize=20)
plt.ylabel('density', fontsize=20)
#plt.grid(color='gray', linewidth=0.5)
#plt.axvline(x=0.45, color='black', linestyle=':', alpha=0.7, label='g to r: 0.45')
#plt.axvline(x=0.8, color='black', linestyle=':', alpha=0.7, label='r to i: 0.8')
#plt.axvline(x=1.2, color='black', linestyle=':', alpha=0.7, label='i to z: 1.2')
#plt.axvline(x=1.45, color='black', linestyle=':', alpha=0.7, label='z to y: 1.45')
# Histograma do ztrue com cor sólida
z = plt.hist(ztrue, bins=50, density=True, color='gray', label='z_true', alpha=0.5)
# Histograma do photoz_mode com transparência
zmode = plt.hist(z_mode, bins=50, density=True, color='red', label='z_phot', alpha=1, histtype='step')#, linestyle='--')
#zmode2 = plt.hist(photoz_mode2, bins=50, density=True, color='blue', label='z_phot magnitudes', alpha=1, histtype='step', linestyle='--')
#zmode1 = plt.hist(photoz_mode1, bins=50, density=True, color='green', label='z_phot magnitudes+cores', alpha=1, histtype='step')
#zmode3 = plt.hist(photoz_mode3, bins=50, density=True, color='red', label='z_phot cores', alpha=1, histtype='step', linestyle='-.' )

# Legenda
plt.legend(fontsize=16)
#plt.title('Minleaf = 30', fontsize=20)
plt.tick_params(axis='both', labelsize=15)
plt.grid(True)
# Salvar em alta qualidade
plt.savefig(f'{path}hist_z_true_mags.pdf', format='pdf', dpi=300, bbox_inches='tight')
#plt.savefig('com_SN.png')
plt.show()

In [ ]:
# Requisitos mínimos:
x_centers = (z[1][:-1] + z[1][1:]) / 2  # Converte as 51 bordas em 50 centros
cs = UnivariateSpline(x_centers, z[0])  # Passa X (50) e Y (50)
cs.set_smoothing_factor(0.2)

In [ ]:
varinf_nz = qp.read(f"NaiveStack_TPZ.hdf5")
#varinf_nz1 = qp.read(f"NaiveStack_TPZ_1.hdf5")
#varinf_nz2 = qp.read(f"NaiveStack_TPZ_2.hdf5")
#varinf_nz4 = qp.read(f"../pkl-files/metricas/NaiveStack_GPZ.hdf5")
fig = plt.figure(figsize=(8, 6))
plt.plot(zgrid,varinf_nz.pdf(zgrid), color = 'red', label = 'z$_{phot}$')#, linestyle='--')
#plt.plot(zgrid,varinf_nz1.pdf(zgrid), color = 'blue', label = 'z$_{phot}$ magnitudes+cores', linestyle='--')
#plt.plot(zgrid,varinf_nz2.pdf(zgrid), color = 'red', label = 'z$_{phot}$ magnitudes')
#plt.plot(zgrid,varinf_nz4.pdf(zgrid), color = 'darkorange', label = 'z$_{phot}$ GPZ', linestyle='-.')
plt.fill_between(zgrid, cs(zgrid), color='gray', alpha=0.5, label='z$_{spec}$') #plt.plot(zgrid,cs(zgrid), color = 'gray', label = 'z$_{spec}$ PDF')
plt.legend(fontsize = 20)
plt.xlabel('z', fontsize=20)
plt.ylabel('p(z)', fontsize=20)
plt.tick_params(axis='both', labelsize=15)
plt.savefig(f'{path}n(z)_mags.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
os.system(f'jupyter nbconvert --to html 04_metrics.ipynb')